# Session 11: Capstone — Full Production Stack IntegrationEnd-to-end demonstration of the complete Ask IRA system: multi-agent orchestration, streaming, caching, guardrails, compliance, portfolio management, and risk assessment.

## 1. System Overview

In [ ]:
import sys
from pathlib import Path

print("""
╔══════════════════════════════════════════════════════════════════╗
║                 Ask IRA — System Overview                    ║
╠══════════════════════════════════════════════════════════════════╣
║  Frontend Layer                                             ║
║  ┌──────────┐ ┌──────────┐ ┌─────┐                    ║
║  │ REST API │ │WebSocket │ │   CLI    │                    ║
║  └────┬────┘ └────┬────┘ └────┬────┘                    ║
║       │            │            │                           ║
║  Middleware Layer (Rate Limit, CORS, Security, Logging)     ║
║       │            │            │                           ║
║  ┌──────────────────────────────────────────────────┐      ║
║  │              Agent Graph (LangGraph)               │      ║
║  │  guard → supervisor → research → analyst → critic │      ║
║  │  → portfolio → risk → writer → compliance → HITL  │      ║
║  └──────────────────────────────────────────────────┘      ║
║                             │                                ║
║  ┌─────────────────┬────────────────────────────────┐      ║
║  │  MCP Servers          │  RAG Pipeline              │      ║
║  │  ├ market_data        │  ├ Query Expansion         │      ║
║  │  ├ sentiment          │  ├ Hybrid Retrieval        │      ║
║  │  ├ macro              │  ├ RRF Fusion              │      ║
║  │  ├ internal_kb        │  └ Cross-Encoder Rerank    │      ║
║  │  └ enterprise_db      │                            │      ║
║  └─────────────────┴────────────────────────────────┘      ║
║                                                              ║
║  Infrastructure: Docker → K8s → Railway → CI/CD              ║
║  Observability: LangSmith + Prometheus + Grafana             ║
╚══════════════════════════════════════════════════════════════════╝
""")


## 2. Quick Smoke Test (No API Keys Required)

In [ ]:
# Test guardrails without LLM
from src.guardrails.input import InputGuardrails
from src.guardrails.output import OutputGuardrails

input_guard = InputGuardrails()
output_guard = OutputGuardrails()

tests = [
    ("What do you think about AAPL?", False, "Legitimate query"),
    ("How can I hack the stock market?", True, "Hacking attempt"),
    ("Tell me about insider trading tips", True, "Insider trading"),
]

print("=== Input Guardrail Tests ===")
for query, expect_blocked, desc in tests:
    result = await input_guard.check(query)
    status = "PASS" if result["blocked"] == expect_blocked else "FAIL"
    print(f"  [{status}] {desc}: blocked={result['blocked']} (expected={expect_blocked})")

In [ ]:
# Test output guardrails
output_tests = [
    ("Comprehensive analysis of AAPL with financial metrics...", False, "Valid report"),
    ("I am not sure about this analysis", True, "Hallucination marker"),
    ("This contains confidential information", True, "Sensitive content"),
]

print("=== Output Guardrail Tests ===")
for report, expect_blocked, desc in output_tests:
    result = await output_guard.check(report)
    status = "PASS" if result["blocked"] == expect_blocked else "FAIL"
    print(f"  [{status}] {desc}: blocked={result['blocked']} (expected={expect_blocked})")

## 3. MCP Server Dispatch

In [ ]:
from src.mcp_servers.registry import MCPRegistry

registry = MCPRegistry(include_enterprise_db=True)
print(f"Registered servers ({len(registry.server_names)}):")
for name in registry.server_names:
    print(f"  └─ {name}")

In [ ]:
# Test MCP dispatch
results = await registry.dispatch_all("Analyze AAPL stock fundamentals")

print("=== MCP Server Results ===")
for name, response in results.items():
    print(f"  [{name}]")
    print(f"    Content: {response.content[:80]}...")
    print(f"    Source: {response.source}")
    print()

In [ ]:
# Test enterprise DB tools
enterprise = registry.get_server("enterprise_db")
result = await enterprise.handle(type("req", (), {"query": "list tables", "context": None})())
print("Enterprise DB tables:")
print(f"  {result.content}")

## 4. RAG Pipeline Test

In [ ]:
from src.rag.vector_store import VectorStore
from src.rag.retrieval import rrf_fusion
from langchain_core.documents import Document

docs = [
    Document(page_content="Apple designs iPhones, Macs, iPads, and provides services", metadata={"ticker": "AAPL"}),
    Document(page_content="Apple services revenue grew 14% to $85B in FY2024", metadata={"ticker": "AAPL"}),
    Document(page_content="Microsoft Azure cloud revenue grew 22% to $95B", metadata={"ticker": "MSFT"}),
]

fused = rrf_fusion([docs[:2], docs[1:]])
print("=== RRF Fusion Result ===")
for d in fused:
    print(f"  {d.page_content[:60]}...")

## 5. Graph Structure Verification

In [ ]:
from src.agents.graph import create_graph

graph = create_graph()

# Verify all nodes exist
nodes = list(graph.nodes.keys())
print("=== Graph Nodes ===")
for i, node in enumerate(nodes, 1):
    print(f"  {i}. {node}")
print(f"\nTotal: {len(nodes)} nodes")

# Verify edges
print("\n=== Graph Edges ===")
edges = list(graph.edges)
for edge in edges[:10]:
    print(f"  {edge[0]} → {edge[1]}")

## 6. Full End-to-End Flow Simulation

In [ ]:
full_flow = '''
QUERY: "Should I invest in Microsoft given current interest rates?"
  │
  └─ guard_input ────────── PASS (no prohibited terms)
  │
  └─ supervisor ────────── Plan: query market_data + macro + sentiment
  │                         RAG: true | Risk profile: moderate
  │
  └─ researcher ───────── [market_data] MSFT: $420, P/E 35x
  │                         [sentiment] Bullish (score 0.35)
  │                         [macro] Fed rate 5.5%, GDP 2.8%
  │                         [internal_kb] MSFT investment framework
  │
  └─ analyst ─────────── Strong financials, AI tailwind,
  │                         rate sensitivity moderate, BUY thesis
  │
  └─ critic_analysis ───── [1st pass] Missing: competitive threats,
  │                         margin trends, VaR estimate
  │   └─ analyst (revised)  [2nd pass] Added: Azure competition,
  │                          margin analysis, VaR: 8.5%
  │
  └─ portfolio_manager ──── Recommended: 8% MSFT in growth portfolio
  │                         Diversification score: 0.72
  │
  └─ risk_assessor ──────── Risk score: 0.35 (moderate)
  │                         VaR (95%): 12.3% monthly
  │                         Risk/Reward: 1.8x favorable
  │
  └─ writer ──────────── 6-section report with exec summary,
  │                         financials, competitive position,
  │                         macro outlook, risk factors, recommendation
  │
  └─ critic_report ─────── Score: 8.5/10
  │                         Suggestion: add ESG section
  │   └─ writer (revised)   ESG analysis appended
  │
  └─ compliance ───────── PASS (all disclaimers present)
  │
  └─ guard_output ─────── PASS (no hallucination markers)
  │
  └─ END ───────────── Final report delivered
                             Confidence: 0.82 | Sections: 7
                             Compliance: ✓ | Risk reviewed: ✓
'''

print(full_flow)


## 7. API Request Example

In [ ]:
api_example = '''
POST /api/v1/query
Content-Type: application/json

{
  "query": "Build a conservative portfolio with MSFT, AAPL, and bonds",
  "session_id": "user-456",
  "risk_profile": "conservative"
}

Response:
{
  "report": "# Portfolio Analysis...\n## Allocation\n...",
  "analysis": "Conservative portfolio with 50% bonds, 30% equities, 15% cash...",
  "portfolio_allocation": {
    "allocation": {"equities": 0.30, "bonds": 0.50, "cash": 0.15, "alternatives": 0.05},
    "recommendation": "Allocate 15% MSFT, 15% AAPL in equity portion...",
    "risk_profile": "conservative"
  },
  "risk_assessment": "Overall risk score: 0.22 (low). VaR (95%): 4.1%...",
  "confidence": 0.88,
  "session_id": "user-456"
}
'''

print(api_example)


## 8. Deployment Verification

In [ ]:
deployment = '''
PRODUCTION CHECKLIST
════════════════

✅ Multi-stage Docker build (slim, non-root, HEALTHCHECK)
✅ Docker Compose with API + Postgres + Redis + Seed
✅ Kubernetes: 3 replicas, HPA (2-10 pods), TLS ingress
✅ Railway: healthcheck, auto-restart, postgres service
✅ CI: lint → pytest (3.11, 3.12) → Docker smoke test
✅ CD: Railway + Docker Hub multi-arch

✅ Structured JSON logging (Logstash/Datadog ready)
✅ Rate limiting (100 req/min per IP)
✅ CORS configurable from env
✅ Security headers (HSTS, XSS, frame options)
✅ Global exception handler (no stack leak)

✅ Input guardrails (blocked patterns, PII, length)
✅ Output guardrails (hallucination markers, sensitive content)
✅ Compliance agent (regulatory keywords, disclaimers)
✅ Human-in-the-loop gate (optional)

✅ WebSocket streaming (stage-by-stage updates)
✅ Response caching (in-memory / Redis)
✅ LangSmith observability (optional)
✅ Prometheus + Grafana monitoring (optional)

Next steps:
  □ Deploy to Railway: railway up --service ask-ira
  □ Add real API keys in production
  □ Configure monitoring alerts
  □ Set up database backups
  □ Load test with k6/locust
'''

print(deployment)


In [ ]:
print("""
╔═════════════════════════════════════════════════╕
║  Ask IRA — Curriculum Complete!                      ║
║                                                      ║
║  Sessions 1-11:                                      ║
║  ├─ AI Overview & Tools                              ║
║  ├─ Python Fundamentals                              ║
║  ├─ NLP, Prompts, Embeddings, RAG                    ║
║  ├─ LangChain & LangGraph                            ║
║  ├─ MCP Servers                                      ║
║  ├─ Advanced RAG                                     ║
║  ├─ AI Agents & Agentic AI                           ║
║  ├─ Production APIs & Streaming                      ║
║  ├─ Docker, CI/CD & Deployment                       ║
║  ├─ Enterprise Security & Multi-Agent Patterns       ║
║  └─ Capstone — Full Production Stack                 ║
║                                                      ║
║  11,000+ lines of code │ 6 agent types │ 5 MCP       ║
║  18 notebooks │ 48+ Python modules │ 37 dependencies ║
╚═════════════════════════════════════════════════╝
""")
